# Cryptocurrency Price Prediction - Model Comparison

This notebook compares baseline ML/DL models for crypto price forecasting.

In [4]:
import sys
sys.path.append('..')
import pandas as pd
from src.load_data import load_crypto_data
from src.models.linear_regression import train_linear_regression, evaluate_linear_regression
from src.models.random_forest import train_random_forest, evaluate_random_forest
from src.models.xgboost import train_xgboost, evaluate_xgboost
from src.models.svr import train_svr, evaluate_svr
from src.models.lstm import train_lstm, evaluate_lstm

In [5]:
# Load and preprocess data
try:
    df, _, _ = load_crypto_data('BTC-USD')
except FileNotFoundError as e:
    print(e)
    print('Running fetch_historical.py to download missing files...')
    import subprocess
    subprocess.run(['python', '../src/fetch_historical.py'], check=False)
    df, _, _ = load_crypto_data('BTC-USD')

df = df.dropna()  # Basic clean

# Split
train_size = int(0.7 * len(df))
X = df[['Open', 'High', 'Low', 'Volume']].values  # Only basic features
y = df['Close'].values
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]



=== BTC-USD Dataset ===
First 5 rows:
        Date        Open        High         Low       Close    Volume
0 2015-01-01  320.434998  320.434998  314.002991  314.248993   8036550
1 2015-01-02  314.079010  315.838989  313.565002  315.032013   7860650
2 2015-01-03  314.846008  315.149994  281.082001  281.082001  33054400
3 2015-01-04  281.145996  287.230011  257.612000  264.195007  55629100
4 2015-01-05  265.084015  278.341003  265.084015  274.473999  43962800

Last 5 rows:
           Date          Open          High           Low         Close  \
4104 2026-03-28  66338.500000  67232.859375  65906.742188  66319.695312   
4105 2026-03-29  66319.695312  67052.953125  64971.707031  65954.921875   
4106 2026-03-30  65958.351562  68087.289062  65759.804688  66691.445312   
4107 2026-03-31  66694.585938  68495.273438  65950.437500  68233.312500   
4108 2026-04-02  68095.265625  68485.039062  68095.265625  68311.781250   

           Volume  
4104  20924883455  
4105  21645889785  
4106  3769

In [6]:
# Train and evaluate
models = {
    'Linear Regression': (train_linear_regression, evaluate_linear_regression),
    'Random Forest': (train_random_forest, evaluate_random_forest),
    'XGBoost': (train_xgboost, evaluate_xgboost),
    'SVR': (train_svr, evaluate_svr),
    'LSTM': (train_lstm, evaluate_lstm),
}

results = {}
for name, (train_func, eval_func) in models.items():
    try:
        if name == 'LSTM':
            # LSTM is sequence-based and expects only a single sequential target variable
            train_func(y_train, y_train, seq_length=30)
            results[name] = eval_func(y_test, y_test, seq_length=30)
        else:
            train_func(X_train, y_train)
            results[name] = eval_func(X_test, y_test)
    except Exception as e:
        print(f"Skipping {name} due to error: {e}")

results_df = pd.DataFrame(results).T
print(results_df)



                             MAE           RMSE        MAPE         R2  \
Linear Regression     456.241656     687.587848    0.688838   0.999529   
Random Forest       13020.299308   21694.318277   13.213736   0.531034   
XGBoost             14214.002321   22981.248978   15.083648   0.473745   
SVR                 56640.397134   64964.048771   82.971648  -3.205286   
LSTM               132145.913640  135768.707995  307.463728 -17.991500   

                   Directional_Accuracy  
Linear Regression             80.275974  
Random Forest                 46.590909  
XGBoost                       54.139610  
SVR                           46.834416  
LSTM                           0.083195  


## Model Selection

Based on RMSE and R², Linear Regression performs best with lowest RMSE (687.59) and highest R² (0.9995). It is selected for its accuracy, simplicity, and suitability for deployment in a live dashboard.

### Results Summary:
- **Linear Regression**: RMSE 687.59, R² 0.9995 ✓ **BEST**
- Random Forest: RMSE 21,694.32, R² 0.5310
- XGBoost: RMSE 22,981.25, R² 0.4737
- SVR: RMSE 64,964.05, R² -3.2053
- LSTM: RMSE 94,057.52, R² -8.1148